# This notebook is used to build a Associations rules using the Opioid Dataset

## These are some of the prerequisites to run this notebook when running in the community edition of databricks

1. Create a New Cluster using the specifications from the quickstart notebook/guide
1. Import the opioid_ARM_class4 dataset file from your system 
1. Attach the cluster you created in Step 1 to this notebook
1. When the cluster changes from <img src="http://docs.databricks.com/_static/images/clusters/cluster-starting.png"/></a> to <img src="http://docs.databricks.com/_static/images/clusters/cluster-running.png"/></a>, you are ready to run this notebook

In [0]:
# Data processing
from pyspark.sql.functions import *

In [0]:
write_path = 'dbfs:/tmp/reproducible_ml_uofl/opioid_ARM_class4.delta'
opioid_ARM_df = spark.read.format('delta').load(write_path)

#display
display(opioid_ARM_df)

PhysID,ITM_1,ITM_2,ITM_3,ITM_4,ITM_5,ITM_6,ITM_7
1003002320,GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_MS,null,null,null
1003009630,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Emergency Medicine,STATE_NY,null,null
1003016270,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Family Practice,STATE_CT,TRAMADOL,null
1003024894,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Orthopedic Surgery,STATE_OH,TRAMADOL,null
1003037979,GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_PA,null,null,null
1003038332,GENDER_M,OXYCODONE,SPCLTY_Internal Medicine,STATE_OR,null,null,null
1003042805,GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_GA,TRAMADOL,null,null
1003043324,GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_TX,TRAMADOL,null,null
1003043480,GENDER_F,HYDROCODONE,SPCLTY_Nurse Practitioner,STATE_FL,null,null,null
1003043928,GENDER_F,OXYCODONE,SPCLTY_Emergency Medicine,STATE_PA,null,null,null


##Now we need to combine all the features/items into one list so we only have two columns: ID, and Items

In [0]:
#opioid_ARM_df2= (opioid_ARM_df.select("ITM_1","ITM_2","ITM_3","ITM_4","ITM_5","ITM_6","ITM_7") )
opioid_ARM_df2= opioid_ARM_df.withColumn("Items", concat_ws(",", col('ITM_1'), col('ITM_2'), col('ITM_3'), col('ITM_4'), col('ITM_5'), col('ITM_6'), col('ITM_7')))
display(opioid_ARM_df2)

PhysID,ITM_1,ITM_2,ITM_3,ITM_4,ITM_5,ITM_6,ITM_7,Items
1003002320,GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_MS,null,null,null,"GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_MS"
1003009630,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Emergency Medicine,STATE_NY,null,null,"GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Emergency Medicine,STATE_NY"
1003016270,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Family Practice,STATE_CT,TRAMADOL,null,"GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Family Practice,STATE_CT,TRAMADOL"
1003024894,GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Orthopedic Surgery,STATE_OH,TRAMADOL,null,"GENDER_M,HYDROCODONE,OXYCODONE,SPCLTY_Orthopedic Surgery,STATE_OH,TRAMADOL"
1003037979,GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_PA,null,null,null,"GENDER_M,HYDROCODONE,SPCLTY_Dentist,STATE_PA"
1003038332,GENDER_M,OXYCODONE,SPCLTY_Internal Medicine,STATE_OR,null,null,null,"GENDER_M,OXYCODONE,SPCLTY_Internal Medicine,STATE_OR"
1003042805,GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_GA,TRAMADOL,null,null,"GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_GA,TRAMADOL"
1003043324,GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_TX,TRAMADOL,null,null,"GENDER_F,HYDROCODONE,SPCLTY_Family Practice,STATE_TX,TRAMADOL"
1003043480,GENDER_F,HYDROCODONE,SPCLTY_Nurse Practitioner,STATE_FL,null,null,null,"GENDER_F,HYDROCODONE,SPCLTY_Nurse Practitioner,STATE_FL"
1003043928,GENDER_F,OXYCODONE,SPCLTY_Emergency Medicine,STATE_PA,null,null,null,"GENDER_F,OXYCODONE,SPCLTY_Emergency Medicine,STATE_PA"


In [0]:
opioid_ARM_df2.printSchema()

root
-- PhysID: integer (nullable = true)
-- ITM_1: string (nullable = true)
-- ITM_2: string (nullable = true)
-- ITM_3: string (nullable = true)
-- ITM_4: string (nullable = true)
-- ITM_5: string (nullable = true)
-- ITM_6: string (nullable = true)
-- ITM_7: string (nullable = true)
-- Items: string (nullable = false)

In [0]:
df = opioid_ARM_df2.select("Items")
df.printSchema()

root
-- Items: string (nullable = false)

In [0]:
df2 = df.select(split(col("Items"),",").alias("Items_List"))
df2.printSchema()
df2.show()

root
-- Items_List: array (nullable = false)
 |-- element: string (containsNull = true)

+--------------------+
 Items_List|
+--------------------+
[GENDER_M, HYDROC...|
[GENDER_M, HYDROC...|
[GENDER_M, HYDROC...|
[GENDER_M, HYDROC...|
[GENDER_M, HYDROC...|
[GENDER_M, OXYCOD...|
[GENDER_F, HYDROC...|
[GENDER_F, HYDROC...|
[GENDER_F, HYDROC...|
[GENDER_F, OXYCOD...|
[GENDER_F, HYDROC...|
[GENDER_M, SPCLTY...|
[GENDER_M, SPCLTY...|
[GENDER_F, OXYCOD...|
[GENDER_M, HYDROC...|
[GENDER_M, HYDROC...|
[GENDER_F, OXYCOD...|
[GENDER_M, HYDROC...|
[GENDER_F, HYDROC...|
[GENDER_F, HYDROC...|
+--------------------+
only showing top 20 rows

In [0]:
from pyspark.ml.fpm import FPGrowth

fpGrowth = FPGrowth(itemsCol="Items_List", minSupport=0.15, minConfidence=0.6)

model = fpGrowth.fit(df2)

# Display frequent itemsets.
#model.freqItemsets.show()

# Display generated association rules.
ar_model = model.associationRules.sort("confidence","antecedent", "consequent")
display(ar_model)

# transform examines the input items against all the association rules and summarize the
# consequents as prediction
#model.transform(df).show()

antecedent,consequent,confidence,lift
"List(TRAMADOL, HYDROCODONE)",List(OXYCODONE),0.6137266023823029,1.2767796204990853
List(TRAMADOL),List(GENDER_M),0.6307854666063621,0.9593881662795319
"List(TRAMADOL, GENDER_M, HYDROCODONE)",List(OXYCODONE),0.6458997392060273,1.3437117125162918
"List(TRAMADOL, HYDROCODONE)",List(GENDER_M),0.6524862923047835,0.992393865484318
List(OXYCODONE),List(GENDER_M),0.6743606775157754,1.025663538656991
List(HYDROCODONE),List(GENDER_M),0.6744324970131422,1.0257727719534595
"List(OXYCODONE, TRAMADOL)",List(GENDER_M),0.6791190409813215,1.032900733933956
"List(OXYCODONE, TRAMADOL, HYDROCODONE)",List(GENDER_M),0.6866913123844732,1.0444177202322058
"List(OXYCODONE, HYDROCODONE)",List(GENDER_M),0.6876646180860404,1.0458980618407083
"List(OXYCODONE, GENDER_M, HYDROCODONE)",List(TRAMADOL),0.7114586658155123,1.3437591082974127


## Assignment 4.2

1. RUn FP Growth with these combinations of minimum support=0.15, 0.10 and minimum confidence = 0.7, 0.75